In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "❌ OPENAI_API_KEY missing from .env file"

print("✅ Environment variables loaded successfully!")

In [ ]:
import pandas as pd

df = pd.read_csv("data/journal_entries.csv")

print(f"📊 Loaded {len(df)} journal entries")
print(f"📋 Columns: {list(df.columns)}")
print("\n🔍 First few rows:")
df.head()

In [ ]:
from sox_copilot.evidence_agent import build_evidence_agent

agent = build_evidence_agent()

print("🤖 Agent built successfully!")

In [ ]:
import json 

control_id = "PAY-002"
period = "2024-07"
csv_path = "data/journal_entries.csv"

print("🚀 Running end-to-end evidence agent...")
res = agent.invoke({
    "control_id": control_id,
    "period": period,
    "csv_path": csv_path,
})

raw = res["output"]
report = json.loads(raw)
report



In [ ]:
required = {"control_id","period","violations_found","violation_entries","policy_summary","population","narrative"}
missing = required - set(report)
assert not missing, f"Missing fields: {missing}"
assert len(report["violation_entries"]) == report["violations_found"]
print("Checks have passed")

In [ ]:
# After you already produced `report` (dict) or `raw` (JSON string) from the evidence agent:

from sox_copilot.reviewer_agent import build_reviewer_agent
import json

# Ensure we have a JSON string for the reviewer input
evidence_json = json.dumps(report)  # or use `raw` directly if you kept it

reviewer = build_reviewer_agent()

print("🤖 Agent built successfully!")



In [ ]:

print("🧮 Running reviewer agent...")
rev_res = reviewer.invoke({
    "evidence": evidence_json,
    "csv_path": "data/journal_entries.csv",
})

rev_raw = rev_res["output"]  # JSON string
print(rev_raw)

# Quick checks
rev = json.loads(rev_raw)
assert set(rev) == {"reviewed_control_id", "period", "evidence_valid", "issues", "review_notes"}
print("✅ Reviewer output shape OK")
rev

In [ ]:
# Test Case: Valid Evidence (from our earlier Evidence Agent run)
print("✅ Test Case: Valid Evidence")
print(f"Evidence valid: {rev['evidence_valid']}")
print(f"Issues found: {rev['issues']}")

assert rev['evidence_valid'] == True, "Valid evidence should pass review"
assert len(rev['issues']) == 0, "Valid evidence should have no issues"
print("✅ Valid evidence test PASSED\n")
